# Requirements

In [ ]:
!pip install flwr[simulation] flwr-datasets

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
PATH = "/Users/tuliopolido/Downloads/WISDM/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Constants

In [ ]:
"""fedrep: A Flower Baseline."""

from enum import Enum

DEFAULT_LOCAL_TRAIN_EPOCHS: int = 10
DEFAULT_FINETUNE_EPOCHS: int = 5
DEFAULT_REPRESENTATION_EPOCHS: int = 1
FEDREP_HEAD_STATE = "fedrep_head_state"

MEAN = {"cifar10": [0.485, 0.456, 0.406], "cifar100": [0.507, 0.487, 0.441]}

STD = {"cifar10": [0.229, 0.224, 0.225], "cifar100": [0.267, 0.256, 0.276]}


class Algorithm(Enum):
    """Algorithm names."""

    FEDREP = "fedrep"
    FEDAVG = "fedavg"


class ModelDatasetName(Enum):
    """Dataset names."""

    CNN_CIFAR_10 = "cnncifar10"
    CNN_CIFAR_100 = "cnncifar100"


# Base Model

In [ ]:
"""fedrep: A Flower Baseline."""

import collections
from abc import ABC, abstractmethod
from typing import Any, Dict, List, OrderedDict, Tuple, Union

import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader

from flwr.common import Context, NDArrays, ParametersRecord, Array


class ModelSplit(ABC, nn.Module):
    """Abstract class for splitting a model into body and head."""

    def __init__(self, model: nn.Module):
        """Initialize the attributes of the model split.

        Args:
            model: dict containing the vocab sizes of the input attributes.
        """
        super().__init__()

        self._body, self._head = self._get_model_parts(model)

    @abstractmethod
    def _get_model_parts(self, model: nn.Module) -> Tuple[nn.Module, nn.Module]:
        """Return the body and head of the model.

        Args:
            model: model to be split into head and body

        Returns
        -------
            Tuple where the first element is the body of the model
            and the second is the head.
        """

    @property
    def body(self) -> nn.Module:
        """Return model body."""
        return self._body

    @body.setter
    def body(self, state_dict: OrderedDict[str, Tensor]) -> None:
        """Set model body.

        Args:
            state_dict: dictionary of the state to set the model body to.
        """
        self._body.load_state_dict(state_dict, strict=True)

    @property
    def head(self) -> nn.Module:
        """Return model head."""
        return self._head

    @head.setter
    def head(self, state_dict: OrderedDict[str, Tensor]) -> None:
        """Set model head.

        Args:
            state_dict: dictionary of the state to set the model head to.
        """
        self._head.load_state_dict(state_dict, strict=True)

    def get_parameters(self) -> NDArrays:
        """Get model parameters.

        Returns
        -------
            Body and head parameters
        """
        return [
            val.cpu().numpy()
            for val in [
                *self.body.state_dict().values(),
                *self.head.state_dict().values(),
            ]
        ]

    def set_parameters(self, state_dict: Dict[str, Tensor]) -> None:
        """Set model parameters.

        Args:
            state_dict: dictionary of the state to set the model to.
        """
        self.load_state_dict(state_dict, strict=False)

    def enable_head(self) -> None:
        """Enable gradient tracking for the head parameters."""
        for param in self._head.parameters():
            param.requires_grad = True

    def enable_body(self) -> None:
        """Enable gradient tracking for the body parameters."""
        for param in self._body.parameters():
            param.requires_grad = True

    def disable_head(self) -> None:
        """Disable gradient tracking for the head parameters."""
        for param in self._head.parameters():
            param.requires_grad = False

    def disable_body(self) -> None:
        """Disable gradient tracking for the body parameters."""
        for param in self._body.parameters():
            param.requires_grad = False

    def forward(self, inputs: Any) -> Any:
        """Forward inputs through the body and the head."""
        return self.head(self.body(inputs))


# pylint: disable=R0902, R0913, R0801
class ModelManager(ABC):
    """Manager for models with Body/Head split."""

    def __init__(
        self,
        context: Context,
        trainloader: DataLoader,
        testloader: DataLoader,
        model_split_class: Any,  # ModelSplit
    ):
        """Initialize the attributes of the model manager.

        Args:
            context: The context of the current run.
            trainloader: Client train dataloader.
            testloader: Client test dataloader.
            model_split_class: Class to be used to split the model into body and head \
                (concrete implementation of ModelSplit).
        """
        super().__init__()
        self.context = context
        self.trainloader = trainloader
        self.testloader = testloader
        self.learning_rate = self.context.run_config.get("learning-rate", 0.01)
        self.momentum = self.context.run_config.get("momentum", 0.5)
        self._model: ModelSplit = model_split_class(self._create_model())

    @abstractmethod
    def _create_model(self) -> nn.Module:
        """Return model to be split into head and body."""

    @property
    def model(self) -> ModelSplit:
        """Return model."""
        return self._model

    def _load_client_state(self) -> None:
        """Load client model head state from context state; used only by FedRep."""
        try:
            # First, check if the fedrep head state is set in the context state.
            if self.context.state.parameters_records.get(FEDREP_HEAD_STATE):
                state_dict = collections.OrderedDict(
                    {
                        k: torch.from_numpy(v.numpy())
                        for k, v in self.context.state.parameters_records[
                            FEDREP_HEAD_STATE
                        ].items()
                    }
                )
                # Second, check if the parameters records have values stored and load
                # the state; this check is useful for the first time the model is
                # tested and the head state might be empty.
                if state_dict:
                    self._model.head.load_state_dict(state_dict)
        except Exception as e:
            print(f"Error loading client state: {e}")

    def _save_client_state(self) -> None:
        """Save client model head state inside context state; used only by FedRep."""
        try:
            # Check if the fedrep head state is set in the context state.
            if FEDREP_HEAD_STATE in self.context.state.parameters_records:
                head_state = self._model.head.state_dict()
                head_state_np = {k: v.detach().cpu().numpy() for k, v in head_state.items()}
                head_state_arr = collections.OrderedDict(
                    {k: Array(v) for k, v in head_state_np.items()}
                )
                head_state_prec = ParametersRecord(head_state_arr)
                self.context.state.parameters_records[FEDREP_HEAD_STATE] = head_state_prec
        except Exception as e:
            print(f"Error saving client state: {e}")

    def train(
        self, device: torch.device
    ) -> Dict[str, Union[List[Dict[str, float]], int, float]]:
        """Train the model maintained in self.model.

        Returns
        -------
            Dict containing the train metrics.
        """
        # try:
        # Load state.
        self._load_client_state()

        num_local_epochs = DEFAULT_LOCAL_TRAIN_EPOCHS
        if "num-local-epochs" in self.context.run_config:
            num_local_epochs = int(self.context.run_config["num-local-epochs"])

        num_rep_epochs = DEFAULT_REPRESENTATION_EPOCHS

        criterion = torch.nn.CrossEntropyLoss()
        weights = [v for k, v in self._model.named_parameters() if "weight" in k]
        biases = [v for k, v in self._model.named_parameters() if "bias" in k]
        optimizer = torch.optim.SGD(
            [
                {"params": weights, "weight_decay": 1e-4},
                {"params": biases, "weight_decay": 0.0},
            ],
            lr=self.learning_rate,
            momentum=self.momentum,
        )
        correct, total = 0, 0
        loss: torch.Tensor = 0.0

        self._model.to(device)
        self._model.train()
        for i in range(num_local_epochs + num_rep_epochs):
            if i < num_local_epochs:
                self._model.disable_body()
                self._model.enable_head()
            else:
                self._model.enable_body()
                self._model.disable_head()
            # for batch in self.trainloader:
            #     images = batch["img"].to(device)
            #     labels = batch["label"].to(device)
            for images, labels in self.trainloader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = self._model(images)
                loss = criterion(outputs, labels)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                total += labels.size(0)
                correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()

        # Save state.
        self._save_client_state()

        return {"loss": loss.item(), "accuracy": correct / total}
        # except Exception as e:
        #     print(f"Error in train(): {e}")
        #     return {"loss": 0.0, "accuracy": 0.0}

    def test(self, device: torch.device) -> Dict[str, float]:
        """Test the model maintained in self.model.

        Returns
        -------
            Dict containing the test metrics.
        """
        # Load state.
        self._load_client_state()

        num_finetune_epochs = DEFAULT_FINETUNE_EPOCHS
        if "num-finetune-epochs" in self.context.run_config:
            num_finetune_epochs = int(self.context.run_config["num-finetune-epochs"])

        if num_finetune_epochs > 0 and self.context.run_config.get(
            "enable-finetune", False
        ):
            optimizer = torch.optim.SGD(self._model.parameters(), lr=self.learning_rate)
            criterion = torch.nn.CrossEntropyLoss()
            self._model.train()
            for _ in range(num_finetune_epochs):
                # for batch in self.trainloader:
                #     images = batch["img"]
                #     labels = batch["label"]
                for images, labels in self.trainloader:
                    images = images.to(device)
                    labels = labels.to(device)
                    outputs = self._model(images)
                    loss = criterion(outputs, labels)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

        criterion = torch.nn.CrossEntropyLoss()
        correct, total, loss = 0, 0, 0.0

        self._model.to(device)
        self._model.eval()
        with torch.no_grad():
            # for batch in self.testloader:
            #     images = batch["img"].to(device)
            #     labels = batch["label"].to(device)
            for images, labels in self.testloader:
                images = images.to(device)
                labels = labels.to(device)
                outputs = self._model(images)
                loss += criterion(outputs, labels).item()
                total += labels.size(0)
                correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()

        return {
            "loss": loss / len(self.testloader.dataset),
            "accuracy": correct / total,
        }

    def train_dataset_size(self) -> int:
        """Return train data set size."""
        return len(self.trainloader.dataset)

    def test_dataset_size(self) -> int:
        """Return test data set size."""
        return len(self.testloader.dataset)

    def total_dataset_size(self) -> int:
        """Return total data set size."""
        return len(self.trainloader.dataset) + len(self.testloader.dataset)


# Models

In [ ]:
"""fedrep: A Flower Baseline."""

from typing import Tuple

import torch
from torch import nn

# pylint: disable=W0223
class CNNCifar10(nn.Module):
    """CNN model for CIFAR10 dataset.

    Refer to
    https://github.com/rahulv0205/fedrep_experiments/blob/main/models/Nets.py
    """

    def __init__(self):
        """Initialize the model."""
        super().__init__()

        # Note that in the official implementation, the body has no BN layers.
        # However, no BN will definitely lead training to collapse.
        self.body = nn.Sequential(
            nn.Conv2d(6, 64, 5),    # 128 -> 124
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(4),        # 124 -> 31

            nn.Conv2d(64, 64, 5),   # 31 -> 27
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(3),        # 27 -> 9

            nn.Flatten(),            # 64 * 9 * 9 = 5184
            nn.Linear(64 * 9 * 9, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
        )

        self.head = nn.Sequential(
            nn.Linear(128, 5)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass of the model."""
        x = self.body(x)
        return self.head(x)


class CNNCifar10ModelSplit(ModelSplit):
    """Split CNNCifar10 model into body and head."""

    def _get_model_parts(self, model: CNNCifar10) -> Tuple[nn.Module, nn.Module]:
        return model.body, model.head


# pylint: disable=R0902, R0913, R0801
class CNNCifar10ModelManager(ModelManager):
    """Manager for models with Body/Head split."""

    def __init__(self, **kwargs):
        """Initialize the attributes of the model manager."""
        super().__init__(model_split_class=CNNCifar10ModelSplit, **kwargs)

    def _create_model(self) -> nn.Module:
        """Return CNNCifar10 model to be split into head and body."""
        return CNNCifar10()


# pylint: disable=W0223
class CNNCifar100(nn.Module):
    """CNN model for 6-channel 128x128 input and 7 classes."""

    def __init__(self):
        """Initialize the model."""
        super().__init__()

        self.body = nn.Sequential(
            nn.Conv2d(6, 64, 5),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 128 -> 62

            nn.Conv2d(64, 128, 5),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 62 -> 29

            nn.Flatten(),
            nn.Linear(128 * 29 * 29, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.6),
        )

        self.head = nn.Sequential(
            nn.Linear(128, 7)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass of the model."""
        x = self.body(x)
        return self.head(x)


class CNNCifar100ModelSplit:
    """Split CNNCifar100 model into body and head."""

    def _get_model_parts(self, model: CNNCifar100) -> Tuple[nn.Module, nn.Module]:
        return model.body, model.head


# pylint: disable=R0902, R0913, R0801
class CNNCifar100ModelManager:
    """Manager for models with Body/Head split."""

    def __init__(self, **kwargs):
        """Initialize the attributes of the model manager."""
        super().__init__(**kwargs)
        self.model_split_class = CNNCifar100ModelSplit
        self.model = self._create_model()

    def _create_model(self) -> CNNCifar100:
        """Return CNNCifar100 model to be split into head and body."""
        return CNNCifar100()


# Strategy

In [ ]:
"""fedrep: A Flower Baseline."""

from flwr.server.strategy import FedAvg


class FedRep(FedAvg):
    """FedRep strategy."""

    def __repr__(self) -> str:
        """Compute a string representation of the strategy."""
        rep = f"FedRep(accept_failures={self.accept_failures})"
        return rep


# Utils

In [ ]:
"""fedrep: A Flower Baseline."""

from typing import Callable, Type, Union

from flwr.common import Context, Parameters
from flwr.server.strategy import FedAvg


def get_create_model_fn(
    context: Context,
) -> tuple[
    Union[Callable[[], CNNCifar10], Callable[[], CNNCifar100]],
    Union[Type[CNNCifar10ModelSplit], Type[CNNCifar100ModelSplit]],
]:
    """Get create model function."""
    model_name = str("cnncifar10")
    if model_name == ModelDatasetName.CNN_CIFAR_10.value:
        split = CNNCifar10ModelSplit

        def create_model() -> CNNCifar10:  # type: ignore
            """Create initial CNNCifar10 model."""
            return CNNCifar10()

    elif model_name == ModelDatasetName.CNN_CIFAR_100.value:
        split = CNNCifar100ModelSplit

        def create_model() -> CNNCifar100:  # type: ignore
            """Create initial CNNCifar100 model."""
            return CNNCifar100()

    else:
        raise NotImplementedError(f"Not a recognized model name {model_name}.")
    return create_model, split


def get_model_manager_class(
    context: Context,
) -> Union[Type[CNNCifar10ModelManager], Type[CNNCifar100ModelManager]]:
    """Depending on the model name type return the corresponding model manager."""
    model_name = str("cnncifar10")
    if model_name.lower() == ModelDatasetName.CNN_CIFAR_10.value:
        model_manager_class = CNNCifar10ModelManager
    elif model_name.lower() == ModelDatasetName.CNN_CIFAR_100.value:
        model_manager_class = CNNCifar100ModelManager  # type: ignore
    else:
        raise NotImplementedError(
            f"Model {model_name} not implemented, please check model name."
        )
    return model_manager_class


def get_server_strategy(
    context: Context, params: Parameters, eval_fn: Callable
) -> Union[FedAvg, FedRep]:
    """Define server strategy based on input algorithm."""
    algorithm = str("fedrep").lower()
    if algorithm == Algorithm.FEDAVG.value:
        strategy = FedAvg
    elif algorithm == Algorithm.FEDREP.value:
        strategy = FedRep
    else:
        raise RuntimeError(f"Unknown algorithm {algorithm}.")

    # Read strategy config
    fraction_fit = float(0.1)
    fraction_evaluate = float(0.1)
    min_available_clients = int(3)

    strategy = strategy(
        fraction_fit=float(fraction_fit),
        fraction_evaluate=fraction_evaluate,
        min_available_clients=min_available_clients,
        initial_parameters=params,
        evaluate_metrics_aggregation_fn=eval_fn,
    )  # type: ignore
    return strategy  # type: ignore


# Dataset

In [ ]:
"""fedrep: A Flower Baseline."""

from typing import Tuple

from flwr_datasets import FederatedDataset
from flwr_datasets.partitioner import PathologicalPartitioner
from flwr_datasets.preprocessor import Merger
from torch.utils.data import DataLoader, Dataset, random_split
import torch
from torchvision import transforms

from flwr.common import Context

FDS = None  # Cache FederatedDataset

def load_wisdm_data(partition_id: int, batch_size: int = 32) -> tuple[DataLoader, DataLoader]:
    """
    Load PAMAP data from a file and return train and test DataLoaders.

    Parameters
    ----------
    partition_id : int
        The partition number to load (file named 'pamap_{partition_id}.pt').
    batch_size : int
        Batch size for DataLoaders.

    Returns
    -------
    Tuple[DataLoader, DataLoader]
        DataLoaders for train and test splits.
    """
    try:
        file_path = f"{PATH}WISDM_processed_user_absFalse_256_64_{int(partition_id + 1)}_rgb_withGyro.pt"

        # Load data (assuming the file contains a dict with 'data' and 'labels')
        dataset_dict = torch.load(file_path)
        data = dataset_dict["data"].float().permute(0, 3, 1, 2)     # convert to float
        labels = dataset_dict["labels"].long()          # ensure labels are int64

        class PamapDataset(Dataset):
            def __init__(self, data, labels):
                self.data = data
                self.labels = labels

            def __len__(self):
                return len(self.data)

            def __getitem__(self, idx):
                x = self.data[idx]
                y = self.labels[idx]
                return x, y


        full_dataset = PamapDataset(data, labels)

        # Split 80% train, 20% test
        train_size = int(0.8 * len(full_dataset))
        test_size = len(full_dataset) - train_size
        train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        return train_loader, test_loader
    except Exception as e:
        print(f"Error in load_pamap_data(): {e}")
        return None, None

def load_wisdm_data2(partition_id: int, batch_size: int = 32) -> tuple[DataLoader, DataLoader]:
    """
    Load WISDM data from a file and return train and test DataLoaders.
    Test data is compiled from users 42 to 51.

    Parameters
    ----------
    partition_id : int
        The partition number to load (file named 'WISDM_processed_user_absFalse_256_64_{partition_id}.pt').
    batch_size : int
        Batch size for DataLoaders.

    Returns
    -------
    Tuple[DataLoader, DataLoader]
        DataLoaders for train and test splits.
    """
    try:
        # Train file path
        train_file_path = f"{PATH}WISDM_processed_user_absFalse_256_64_{partition_id + 1}_rgb_withGyro.pt"

        # Load train data
        train_dataset_dict = torch.load(train_file_path)
        train_data = train_dataset_dict["data"].float().permute(0, 3, 1, 2)
        train_labels = train_dataset_dict["labels"].long()

        # Compile test data from multiple user files
        test_files = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]
        test_data_list = []
        test_labels_list = []

        for i in test_files:
            test_file_path = f"{PATH}WISDM_processed_user_absFalse_256_64_{i}_rgb_withGyro.pt"
            test_dataset_dict = torch.load(test_file_path)
            test_data_list.append(test_dataset_dict["data"].float().permute(0, 3, 1, 2))
            test_labels_list.append(test_dataset_dict["labels"].long())

        # Concatenate all test data
        test_data = torch.cat(test_data_list, dim=0)
        test_labels = torch.cat(test_labels_list, dim=0)

        # Dataset class
        class PamapDataset(Dataset):
            def __init__(self, data, labels):
                self.data = data
                self.labels = labels

            def __len__(self):
                return len(self.data)

            def __getitem__(self, idx):
                return self.data[idx], self.labels[idx]

        # DataLoaders
        train_loader = DataLoader(PamapDataset(train_data, train_labels), batch_size=batch_size, shuffle=True)
        test_loader = DataLoader(PamapDataset(test_data, test_labels), batch_size=batch_size, shuffle=False)

        return train_loader, test_loader

    except Exception as e:
        print(f"Error in load_wisdm_data2(): {e}")
        return None, None

def load_data(
    partition_id: int, num_partitions: int, context: Context
) -> Tuple[DataLoader, DataLoader]:
    """Split the data and return training and testing data for the specified partition.

    Parameters
    ----------
    partition_id : int
        Partition number for which to retrieve the corresponding data.
    num_partitions : int
        Total number of partitions.
    context: Context
        the context of the current run.

    Returns
    -------
    data : Tuple[DataLoader, DataLoader]
        A tuple with the training and testing data for the current partition_id.
    """
    try:
        batch_size = int(50)
        dataset_name = str("cifar10").lower()
        dataset_split_num_classes = int(2)
        dataset_split_seed = int(42)
        dataset_split_fraction = float(0.83)

        # - you can define your own data transformation strategy here -
        # These transformations are from the official repo
        train_data_transform = transforms.Compose(
            [
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(MEAN[dataset_name], STD[dataset_name]),
            ]
        )
        test_data_transform = transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.Normalize(MEAN[dataset_name], STD[dataset_name]),
            ]
        )

        use_fine_label = False
        if dataset_name == "cifar100":
            use_fine_label = True

        partitioner = PathologicalPartitioner(
            num_partitions=num_partitions,
            partition_by="fine_label" if use_fine_label else "label",
            num_classes_per_partition=dataset_split_num_classes,
            class_assignment_mode="random",
            shuffle=True,
            seed=dataset_split_seed,
        )

        global FDS  # pylint: disable=global-statement
        if FDS is None:
            FDS = FederatedDataset(
                dataset=dataset_name,
                partitioners={"all": partitioner},
                preprocessor=Merger({"all": ("train", "test")}),
            )

        def apply_train_transforms(batch: Dataset) -> Dataset:
            """Apply transforms for train data to the partition from FederatedDataset."""
            batch["img"] = [train_data_transform(img) for img in batch["img"]]
            if use_fine_label:
                batch["label"] = batch["fine_label"]
            return batch

        def apply_test_transforms(batch: Dataset) -> Dataset:
            """Apply transforms for test data to the partition from FederatedDataset."""
            batch["img"] = [test_data_transform(img) for img in batch["img"]]
            if use_fine_label:
                batch["label"] = batch["fine_label"]
            return batch

        partition = FDS.load_partition(partition_id, split="all")

        partition_train_test = partition.train_test_split(
            train_size=dataset_split_fraction, shuffle=True, seed=dataset_split_seed
        )

        trainset = partition_train_test["train"].with_transform(apply_train_transforms)
        testset = partition_train_test["test"].with_transform(apply_test_transforms)

        trainloader = DataLoader(trainset, batch_size, shuffle=True)
        testloader = DataLoader(testset, batch_size)

        return trainloader, testloader
    except Exception as e:
        print(f"Error in load_data(): {e}")
        return None, None


# MAIN

In [ ]:
"""fedprox: A Flower Baseline."""

import toml

from collections import OrderedDict
from typing import Dict, List, Tuple, Union

import torch

from flwr.client import ClientApp, NumPyClient
from flwr.client.client import Client
from flwr.common import Context, NDArrays, ParametersRecord, Scalar

from flwr.simulation import run_simulation

import json
import os
import time
from typing import Dict, List, Tuple

from flwr.common import Context, Metrics, ndarrays_to_parameters
from flwr.server import ServerApp, ServerAppComponents, ServerConfig

RESULTS_FILE = "result-wisdm-{}.json"

import sys, os

NUM_PARTITIONS = 41
NUM_ROUNDS = 100

class DotDict(dict):
    def __getattr__(self, key):
        val = self[key]
        if isinstance(val, dict):
            return DotDict(val)
        return val

# Usage:
CONFIG = DotDict(toml.load(open(f"{PATH}pyproject.toml")))
print(CONFIG.tool.flwr.app.config.algorithm)



class BaseClient(NumPyClient):
    """Implementation of Federated Averaging (FedAvg) Client."""

    # pylint: disable=R0913
    def __init__(
        self, model_manager: Union[CNNCifar10ModelManager, CNNCifar100ModelManager]
    ):
        """Initialize client attributes.

        Args:
            model_manager: the model manager object
        """
        try:
            super().__init__()
            self.model_manager = model_manager

            self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
            print(f"DEVICE:{self.device}")
        except Exception as e:
            print(f"Error in __init__(): {e}")

    def get_parameters(self, config: Dict[str, Scalar]) -> NDArrays:
        """Return the current local model parameters."""
        try:
            return self.model_manager.model.get_parameters()
        except Exception as e:
            print(f"Error in get_parameters(): {e}")

    def set_parameters(self, parameters: NDArrays, evaluate: bool = False) -> None:
        """Set the local model parameters to the received parameters.

        Args:
            parameters: parameters to set the model to.
            evaluate: whether to evaluate or not.
        """
        try:
            _ = evaluate
            model_keys = [
                k
                for k in self.model_manager.model.state_dict().keys()
                if k.startswith("_body") or k.startswith("_head")
            ]
            params_dict = zip(model_keys, parameters)
            state_dict = OrderedDict({k: torch.tensor(v) for k, v in params_dict})
            self.model_manager.model.set_parameters(state_dict)
        except Exception as e:
            print(f"Error in set_parameters(): {e}")

    def perform_train(self) -> Dict[str, Union[List[Dict[str, float]], int, float]]:
        """Perform local training to the whole model.

        Returns
        -------
            Dict with the train metrics.
        """
        self.model_manager.model.enable_body()
        self.model_manager.model.enable_head()

        return self.model_manager.train(self.device)

    def fit(
        self, parameters: NDArrays, config: Dict[str, Scalar]
    ) -> Tuple[NDArrays, int, Dict[str, Union[bool, bytes, float, int, str]]]:
        """Train the provided parameters using the locally held dataset.

        Args:
            parameters: The current (global) model parameters.
            config: configuration parameters for training sent by the server.

        Returns
        -------
            Tuple containing the locally updated model parameters, \
                the number of examples used for training and \
                the training metrics.
        """
        self.set_parameters(parameters)
        self.perform_train()

        return self.get_parameters(config), self.model_manager.train_dataset_size(), {}


    def evaluate(
        self, parameters: NDArrays, config: Dict[str, Scalar]
    ) -> Tuple[float, int, Dict[str, Union[bool, bytes, float, int, str]]]:
            """Evaluate the provided global parameters using the locally held dataset.

            Args:
                parameters: The current (global) model parameters.
                config: configuration parameters for training sent by the server.

            Returns
            -------
            Tuple containing the test loss, \
                    the number of examples used for evaluation and \
                    the evaluation metrics.
            """
            try:
                self.set_parameters(parameters, evaluate=True)

                # Test the model
                test_results = self.model_manager.test(self.device)

                return (
                    test_results.get("loss", 0.0),
                    self.model_manager.test_dataset_size(),
                    {k: v for k, v in test_results.items() if not isinstance(v, (dict, list))},
                )
            except Exception as e:
                print(f"Error in evaluate(): {e}")


class FedRepClient(BaseClient):
    """Implementation of Federated Personalization (FedRep) Client."""

    def get_parameters(self, config: Dict[str, Scalar]) -> NDArrays:
        """Return the current local body parameters."""
        try:
            return [
                val.cpu().numpy()
                for val in self.model_manager.model.body.state_dict().values()
            ]
        except Exception as e:
            print(f"Error in get_parameters(): {e}")

    def set_parameters(self, parameters: NDArrays, evaluate: bool = False) -> None:
        """Set the local body parameters to the received parameters.

        Args:
            parameters: parameters to set the body to.
            evaluate: whether the client is evaluating or not.
        """
        try:
            model_keys = [
                k
                for k in self.model_manager.model.state_dict().keys()
                if k.startswith("_body")
            ]

            if not evaluate:
                # Only update client's local head if it hasn't trained yet
                model_keys.extend(
                    [
                        k
                        for k in self.model_manager.model.state_dict().keys()
                        if k.startswith("_head")
                    ]
                )

            state_dict = OrderedDict(
                (k, torch.from_numpy(v)) for k, v in zip(model_keys, parameters)
            )

            self.model_manager.model.set_parameters(state_dict)
        except Exception as e:
            print(f"Error in set_parameters(): {e}")


def client_fn(context: Context) -> Client:
    """Construct a Client that will be run in a ClientApp."""
    try:
        model_manager_class = get_model_manager_class(context)
        algorithm = str("fedrep").lower()
        partition_id = int(context.node_config["partition-id"])
        num_partitions = int(NUM_PARTITIONS)
        # trainloader, valloader = load_data(
        #     partition_id, num_partitions, context
        # )  # load the data
        trainloader, valloader = load_wisdm_data2(partition_id)  # load the data

        if algorithm == Algorithm.FEDAVG.value:
            client_class = BaseClient
        elif algorithm == Algorithm.FEDREP.value:
            # This state variable will only be used by the FedRep algorithm.
            # We only need to initialize once, since client_fn will be called
            # again at every invocation of the ClientApp.
            if FEDREP_HEAD_STATE not in context.state.parameters_records:
                context.state.parameters_records[FEDREP_HEAD_STATE] = ParametersRecord()
            client_class = FedRepClient
        else:
            raise RuntimeError(f"Unknown algorithm {algorithm}.")

        model_manager_obj = model_manager_class(
            context=context, trainloader=trainloader, testloader=valloader
        )

        # Return client object.
        client = client_class(model_manager_obj).to_client()
        return client
    except Exception as e:
        print(f"Error in client_fn(): {e}")
        return None




def config_json_file(context: Context) -> None:
    """Initialize the json file and write the run configurations."""
    # Initialize the execution results directory.
    res_save_path = f"{PATH}"
    if not os.path.exists(res_save_path):
        os.makedirs(res_save_path)
    res_save_name = time.strftime("%Y-%m-%d-%H-%M-%S")
    # Set the date and full path of the file to save the results.
    global RESULTS_FILE  # pylint: disable=global-statement
    RESULTS_FILE = RESULTS_FILE.format(res_save_name)
    RESULTS_FILE = f"{res_save_path}/{RESULTS_FILE}"
    data = {
        "run_config": dict(context.run_config.items()),
        "round_res": [],
    }
    with open(RESULTS_FILE, "w+", encoding="UTF-8") as fout:
        json.dump(data, fout, indent=4)


def write_res(new_res: Dict[str, float]) -> None:
    """Load the json file, append result and re-write json collection."""
    with open(RESULTS_FILE, "r", encoding="UTF-8") as fin:
        data = json.load(fin)
    data["round_res"].append(new_res)

    # Write the updated data back to the JSON file
    with open(RESULTS_FILE, "w", encoding="UTF-8") as fout:
        json.dump(data, fout, indent=4)


def evaluate_metrics_aggregation_fn(eval_metrics: List[Tuple[int, Metrics]]) -> Metrics:
    """Weighted metrics evaluation."""
    weights, accuracies, losses = [], [], []
    for num_examples, metric in eval_metrics:
        weights.append(num_examples)
        accuracies.append(float(metric["accuracy"]) * num_examples)
        losses.append(float(metric["loss"]) * num_examples)
    accuracy = sum(accuracies) / sum(weights)
    loss = sum(losses) / sum(weights)
    write_res({"accuracy": accuracy, "loss": loss})
    return {"accuracy": accuracy}


def server_fn(context: Context) -> ServerAppComponents:
    """Construct components that set the ServerApp behaviour."""
    config_json_file(context)
    # Read from config

    num_rounds = NUM_ROUNDS

    # Initialize model parameters
    create_model_fn, split_class = get_create_model_fn(context)
    net = split_class(create_model_fn())
    parameters = ndarrays_to_parameters(net.get_parameters())

    # Define strategy
    strategy = get_server_strategy(
        context=context, params=parameters, eval_fn=evaluate_metrics_aggregation_fn
    )
    config = ServerConfig(num_rounds=int(num_rounds))

    return ServerAppComponents(strategy=strategy, config=config)

# Create ServerApp
server_app = ServerApp(server_fn=server_fn)
# Flower ClientApp
client_app = ClientApp(client_fn)

run_simulation(
    server_app=server_app, client_app=client_app, num_supernodes=NUM_PARTITIONS
)

fedrep


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      Starting Flower ServerApp, config: num_rounds=50, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Using initial global parameters provided by strategy
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 4 clients (out of 41)
(pid=15467) 2025-09-30 22:57:18.303749: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
(pid=15467) WARNING: All log messages before absl::InitializeLog() 

(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu


INFO :      aggregate_fit: received 4 results and 0 failures
INFO :      configure_evaluate: strategy sampled 4 clients (out of 41)


(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu


...

INFO :      aggregate_evaluate: received 4 results and 0 failures
INFO :      
INFO :      [ROUND 50]
INFO :      configure_fit: strategy sampled 4 clients (out of 41)


(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu


INFO :      aggregate_fit: received 4 results and 0 failures
INFO :      configure_evaluate: strategy sampled 4 clients (out of 41)


(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu
(ClientAppActor pid=15467) DEVICE:cpu


INFO :      aggregate_evaluate: received 4 results and 0 failures
INFO :      
INFO :      [SUMMARY]
INFO :      Run finished 50 round(s) in 16555.58s
INFO :      	History (loss, distributed):
INFO :      		round 1: 0.00610218831413501
INFO :      		round 2: 0.006029793313024817
INFO :      		round 3: 0.006717000874011098
INFO :      		round 4: 0.003048279382464652
INFO :      		round 5: 0.0030566508397208273
INFO :      		round 6: 0.010108601928034143
INFO :      		round 7: 0.004927178671759729
INFO :      		round 8: 0.008552880861758682
INFO :      		round 9: 0.0027813061527479943
INFO :      		round 10: 0.0029739164825956766
INFO :      		round 11: 0.0028220294137812547
INFO :      		round 12: 0.0019380844125801043
INFO :      		round 13: 0.003793083656755452
INFO :      		round 14: 0.0010103893299536729
INFO :      		round 15: 0.001501453249098582
INFO :      		round 16: 0.0017829628533847328
INFO :      		round 17: 0.0011847646272980382
INFO :      		round 18: 0.002411699103430618